###Requirement on transformations:
####1. Answer the questions using the dev.spark_db.sf_fire_calls
What are the top 3 zip codes that accounted for most calls from the SF fire calls table?

In [0]:
%sql

SELECT calltype, Zipcode, count(*) as Count
FROM dev.spark_db.sf_fire_calls
WHERE CallType IS NOT NULL
GROUP BY CallType, Zipcode 
ORDER BY count desc
limit 3


calltype,Zipcode,Count
Medical Incident,94102,16130
Medical Incident,94103,14775
Medical Incident,94110,9995


#### 2. Using Spark API to mimic the above SQL query to print the results

2.1 Create a DataFrame reading data from the table

In [0]:
# Read data and create a dataframe

fire_df = spark.read.table("dev.spark_db.sf_fire_calls")

2.2 Performing the Transformations on the created dataframe

In [0]:
# Apply necessary transformations
df_1 = fire_df.select("CallType","Zipcode")
df_2 = df_1.where("CallType is not null")
df_3 = df_2.groupBy("CallType","Zipcode").count()
df_4 = df_3.orderBy("count", ascending=False).limit(3)



In [0]:
df_4.show()

+----------------+-------+-----+
|        CallType|Zipcode|count|
+----------------+-------+-----+
|Medical Incident|  94102|16130|
|Medical Incident|  94103|14775|
|Medical Incident|  94110| 9995|
+----------------+-------+-----+



In [0]:
df_4.explain(mode="extended")

== Parsed Logical Plan ==
'GlobalLimit 3
+- 'LocalLimit 3
   +- 'Sort ['count DESC NULLS LAST], true
      +- 'Aggregate ['CallType, 'Zipcode], ['CallType, 'Zipcode, 'count(1) AS count#14756]
         +- 'Filter isnotnull('CallType)
            +- 'Project ['CallType, 'Zipcode]
               +- 'UnresolvedRelation [dev, spark_db, sf_fire_calls], [], false

== Analyzed Logical Plan ==
CallType: string, Zipcode: string, count: bigint
GlobalLimit 3
+- LocalLimit 3
   +- Sort [count#14756L DESC NULLS LAST], true
      +- Aggregate [CallType#14788, Zipcode#14795], [CallType#14788, Zipcode#14795, count(1) AS count#14756L]
         +- Filter isnotnull(CallType#14788)
            +- Project [CallType#14788, Zipcode#14795]
               +- SubqueryAlias dev.spark_db.sf_fire_calls
                  +- Relation dev.spark_db.sf_fire_calls[CallNumber#14785,UnitID#14786,IncidentNumber#14787,CallType#14788,CallDate#14789,WatchDate#14790,CallFinalDisposition#14791,AvailableDtTm#14792,Address#14793,C

2. Dataframes are immutable\
You can see or use any intermediate dataframe

In [0]:
df_4.display()

CallType,Zipcode,count
Medical Incident,94102,16130
Medical Incident,94103,14775
Medical Incident,94110,9995


3. Every Transformation returns a dataframe
4. Dataframe offers composable API\
  Example of composable API

In [0]:
result_df = (
    fire_df.select("CallType", "Zipcode")
            .where("CallType is not null")
            .groupBy("CallType", "Zipcode")
            .count()
            .orderBy("count", ascending=False)
            .limit(3)
)

result_df.display()

CallType,Zipcode,count
Medical Incident,94102,16130
Medical Incident,94103,14775
Medical Incident,94110,9995
